# Разбор Узкой Границы `O/B` В Coarse-Классификации

Цель:
- Выделить narrow boundary source: true `O/B`, `quality_state = pass`, `teff_gspphot >= 10000 K`.
- Проверить, насколько асимметрична граница `O -> B` против `B -> O`.
- Посмотреть полные `P(O)` и `P(B)`, а не только top-label.
- Зафиксировать evidence до любого narrow retrain или class-weight policy для `O`.

In [1]:
# Setup: repo root, sys.path и display-настройки.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
from IPython.display import display

def find_repo_root(start_path: Path) -> Path:
    current_path = start_path.resolve()
    while current_path != current_path.parent:
        if (current_path / 'src').exists() and (current_path / '.git').exists():
            return current_path
        current_path = current_path.parent
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 160)

In [2]:
# Импортируем review helpers после добавления src в sys.path.
from exohost.reporting.archive_research.coarse_ob_boundary_review import (
    CoarseOBBoundaryReviewConfig,
    load_coarse_ob_boundary_review_bundle_from_env,
    build_boundary_membership_summary_frame,
    build_boundary_true_class_summary_frame,
    build_boundary_predicted_class_summary_frame,
    build_boundary_confusion_frame,
    build_boundary_probability_summary_frame,
    build_boundary_physics_summary_frame,
    build_high_confidence_o_to_b_preview_frame,
    build_high_confidence_b_to_o_preview_frame,
)

## План

1. Загружаем narrow boundary source только для true `O/B` с physics-cut `teff_gspphot >= 10000 K`.
2. Сначала смотрим, сколько true `O` и true `B` вообще попадает в этот slice.
3. Потом разбираем, во что coarse-model переводит этот boundary source.
4. Отдельно смотрим `P(O)` и `P(B)` на true `O` и true `B`.
5. После этого проверяем, насколько проблема асимметрична: `O -> B` против `B -> O`.
6. В конце фиксируем, нужен ли следующий пакет уже под narrow retrain/class-weighting.

In [3]:
# Конфигурация notebook.
COARSE_MODEL_RUN_DIR = REPO_ROOT / 'artifacts/models/gaia_id_coarse_classification__hist_gradient_boosting__2026_03_28_215003_509969'
BOUNDARY_CONFIG = CoarseOBBoundaryReviewConfig(quality_state='pass', teff_min_k=10000.0)
SOURCE_LIMIT = None
PREVIEW_ROWS = 20

if not COARSE_MODEL_RUN_DIR.exists():
    raise FileNotFoundError(COARSE_MODEL_RUN_DIR)


In [4]:
# Загружаем narrow `O/B` boundary bundle.
bundle = load_coarse_ob_boundary_review_bundle_from_env(
    coarse_model_run_dir=COARSE_MODEL_RUN_DIR,
    config=BOUNDARY_CONFIG,
    source_limit=SOURCE_LIMIT,
    dotenv_path='.env',
)

display(build_boundary_membership_summary_frame(bundle))

,quality_state,teff_min_k,n_rows_boundary_source,n_rows_scored
0,pass,10000.0,8300,8300


In [5]:
# Сколько true `O` и true `B` входит в boundary source.
display(build_boundary_true_class_summary_frame(bundle.source_df))
display(build_boundary_physics_summary_frame(bundle.source_df))

,true_spectral_class,n_rows,share
0,B,7112,0.856867
1,O,1188,0.143133


,true_spectral_class,n_rows,median_teff_gspphot,median_logg_gspphot,median_bp_rp,median_radius_flame
0,B,7112,12311.6575,3.7899,0.198108,3.530580
1,O,1188,15613.8865,3.6724,0.477677,4.778071


In [6]:
# Во что coarse-model переводит narrow boundary source.
display(build_boundary_predicted_class_summary_frame(bundle.scored_df))
display(build_boundary_confusion_frame(bundle.scored_df))

,coarse_predicted_label,n_rows,share
0,B,8279,0.99747
1,A,21,0.00253


,true_spectral_class,coarse_predicted_label,n_rows,share_within_true_class
0,B,B,7091,0.997047
1,B,A,21,0.002953
2,O,B,1188,1.000000


In [7]:
# Что происходит с полными `P(O)` и `P(B)` на true `O` и true `B`.
display(build_boundary_probability_summary_frame(bundle.scored_df))

,true_spectral_class,n_rows,median_probability__O,median_probability__B,mean_probability__O,mean_probability__B
0,B,7112,0.000017,0.999842,0.000031,0.997244
1,O,1188,0.000016,0.999846,0.000084,0.999749


In [8]:
# Самые уверенные асимметричные случаи.
display(build_high_confidence_o_to_b_preview_frame(bundle.scored_df, top_n=PREVIEW_ROWS))
display(build_high_confidence_b_to_o_preview_frame(bundle.scored_df, top_n=PREVIEW_ROWS))

,source_id,spectral_class,coarse_predicted_label,coarse_probability__O,coarse_probability__B,coarse_probability_max,coarse_probability_margin,teff_gspphot,logg_gspphot,mh_gspphot,bp_rp,parallax,parallax_over_error,ruwe,radius_flame
0,6233150491818490880,O,B,0.000013,0.999863,0.999863,0.999836,17983.480,3.8043,0.0482,0.090590,0.138179,6.129621,0.968518,5.596548
1,5258822695275015936,O,B,0.000015,0.999854,0.999854,0.999825,15240.279,3.3739,0.2620,0.977855,0.363321,29.931099,0.769798,8.168468
2,2163856681813296768,O,B,0.000015,0.999853,0.999853,0.999824,15004.000,3.3951,0.0914,0.907979,0.269155,28.054030,0.772332,8.384238
3,6057434652418534144,O,B,0.000016,0.999852,0.999852,0.999823,13760.666,3.5602,0.3592,0.499101,0.212685,14.561953,0.805861,8.201930
4,2164092393934703872,O,B,0.000016,0.999852,0.999852,0.999823,15003.062,3.4547,0.0920,0.845365,0.294649,22.302872,0.936608,8.374869
5,5333054569254979072,O,B,0.000016,0.999852,0.999852,0.999823,15004.313,3.4220,0.0708,0.589947,0.249087,17.533644,0.875767,8.246414
6,258576895448200832,O,B,0.000016,0.999850,0.999850,0.999821,11149.087,3.4565,0.0173,1.005429,0.441580,22.317532,0.970959,8.716607
7,5258869390156104704,O,B,0.000016,0.999850,0.999850,0.999820,16877.060,3.9903,-0.2689,0.378989,0.336944,24.205150,0.854912,4.572591
8,5600744904276023296,O,B,0.000016,0.999850,0.999850,0.999820,15007.395,3.9943,0.0314,0.777427,0.276553,19.130670,1.048306,2.888663
9,5203319520096719360,O,B,0.000016,0.999849,0.999849,0.999819,16377.575,3.3548,0.2894,0.193219,0.113841,8.954993,1.021717,4.105176


,source_id,spectral_class,coarse_predicted_label,coarse_probability__O,coarse_probability__B,coarse_probability_max,coarse_probability_margin,teff_gspphot,logg_gspphot,mh_gspphot,bp_rp,parallax,parallax_over_error,ruwe,radius_flame


## Опорные Источники

- [Gaia DR3 documentation index](https://gea.esac.esa.int/archive/documentation/GDR3/)
- [Gaia DR3 astrophysical_parameters semantics](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_astrophysical_parameter_tables/ssec_dm_astrophysical_parameters.html)
- [Astrophysical parameters associated to hot stars in Gaia DR3](https://www.aanda.org/articles/aa/full_html/2023/06/aa43709-22/aa43709-22.html)
- [NASA spectral classification overview](https://asd.gsfc.nasa.gov/archive/star_class/spectral_classification.html)
- [ESA star types overview](https://www.esa.int/Science_Exploration/Space_Science/Star_types)

## Следующие Шаги

- Если `O -> B` остается полностью односторонним, следующий пакет должен разбирать уже train-time representation hottest `O` tail и class-weight policy для `O`.
- Если появится заметный `B -> O` хвост, тогда проблема может быть шире и потребует другого boundary-design.
- `priority` и `quality_gate` здесь не трогаем: этот notebook отвечает только за узкую границу `O/B` в coarse-stage.